In [1]:
!pip install openai fastapi uvicorn langchain pydantic

Defaulting to user installation because normal site-packages is not writeable
  Using cached fastapi-0.135.1-py3-none-any.whl.metadata (30 kB)
  Using cached uvicorn-0.42.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langchain-1.2.13-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.2.20-py3-none-any.whl.metadata (4.4 kB)
  Using cached langgraph-1.1.3-py3-none-any.whl.metadata (7.4 kB)
  Using cached langsmith-0.7.22-py3-none-any.whl.metadata (15 kB)
Using cached fastapi-0.135.1-py3-none-any.whl (116 kB)
Using cached uvicorn-0.42.0-py3-none-any.whl (68 kB)
Using cached langchain-1.2.13-py3-none-any.whl (112 kB)
Using cached langchain_core-1.2.20-py3-none-any.whl (504 kB)
Using cached langgraph-1.1.3-py3-none-any.whl (168 kB)
Using cached langsmith-0.7.22-py3-none-any.whl (359 kB)

  Attempting uninstall: langsmith

    Found existing installation: langsmith 0.1.147

    Uninstalling langsmith-0.1.147:

      Successfully uninstalled langsmith-0.1.147

   ------ 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 1.5.0 requires numpy>=1.22.5, which is not installed.
langchain-chroma 1.1.0 requires numpy>=2.1.0; python_version >= "3.13", which is not installed.
chromadb 1.5.0 requires tqdm>=4.65.0, but you have tqdm 4.64.1 which is incompatible.
fastmcp 3.1.1 requires packaging>=24.0, but you have packaging 23.2 which is incompatible.


Imports & Client

In [2]:
from openai import OpenAI
import json
client = OpenAI()

Intent & Entity Extraction

In [3]:
def extract_intent_entities(user_input: str):
    prompt = f"""
    Extract intent and entities from: "{user_input}"
    Return JSON with keys: intent, entities.
    """
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(response.choices[0].message.content)
extract_intent_entities("Book compliance training next week")

{'intent': 'BookTraining',
 'entities': [{'training': 'compliance training', 'time': 'next week'}]}

Context Manager

In [4]:
class ContextManager:
    def __init__(self):
        self.state = {}

    def update_context(self, user_id, intent, entities):
        if user_id not in self.state:
            self.state[user_id] = []
        self.state[user_id].append({"intent": intent, "entities": entities})
        return self.state[user_id]

context = ContextManager()

Router Layer

In [5]:
def route_intent(intent, entities):
    if intent == "book_training":
        return {"status": "success", "details": f"Training booked: {entities}"}
    elif intent == "update_leave":
        return {"status": "success", "details": f"Leave updated: {entities}"}
    else:
        return {"status": "unknown_intent"}

 Response Generator

In [6]:
def generate_response(intent, entities, backend_result):
    if backend_result["status"] == "success":
        return f"✅ Your request for {intent} with {entities} has been processed."
    else:
        return f"⚠️ I couldn’t process your request. Please refine your input."

Full Pipeline Function

In [7]:
def clu_pipeline(user_id: str, message: str):
    parsed = extract_intent_entities(message)
    context_state = context.update_context(user_id, parsed["intent"], parsed["entities"])
    backend_result = route_intent(parsed["intent"], parsed["entities"])
    response = generate_response(parsed["intent"], parsed["entities"], backend_result)
    return {"response": response, "context": context_state}

# Example Run
clu_pipeline("user123", "I want to book compliance training next week")

{'response': "✅ Your request for book_training with {'training_type': 'compliance', 'time': 'next week'} has been processed.",
 'context': [{'intent': 'book_training',
   'entities': {'training_type': 'compliance', 'time': 'next week'}}]}